In [ ]:
!pip install langchain_community langchain_text_splitters faiss-gpu-cu12 -q
!pip install pypdf -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 793.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 125.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 MB 12.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.0/346.0 kB 2.2 MB/s eta 0:00:00


In [ ]:
#Loaderclass
class DocumentLoader:

    def __init__(self, loader_cls):
        self.loader_cls = loader_cls

    def load(self, path):
        loader = self.loader_cls(path)
        return loader.load()

In [ ]:
#chunkerclass
from langchain_text_splitters import RecursiveCharacterTextSplitter

class Chunker:

    def __init__(
        self,
        child_chunk_size=400,
        child_chunk_overlap=100,
        parent_chunk_size=1000,
        parent_chunk_overlap=200
    ):
        self.child_splitter = RecursiveCharacterTextSplitter(
            chunk_size=child_chunk_size,
            chunk_overlap=child_chunk_overlap
        )
        self.parent_splitter = RecursiveCharacterTextSplitter(
            chunk_size=parent_chunk_size,
            chunk_overlap=parent_chunk_overlap
        )

#----------------------parent-child implimentation--------------------------

    def split(self, documents):
        parent_chunks = self.parent_splitter.split_documents(documents)
        parents = {}
        children = {}
        child_to_parent = {}
        parent_id, child_id = 0, 0

        for parent in parent_chunks:
            parents[parent_id] = parent
            child_doc = self.child_splitter.split_documents([parent])

            for child in child_doc:
                children[child_id] = child
                child_to_parent[child_id] = parent_id
                child_id += 1

            parent_id += 1

        return parents, children, child_to_parent

In [ ]:
#embedder class
import numpy as np

class Embedder:

    def __init__(self, emb_model):
        self.model = emb_model

    def embed_docs(self, texts):
        emb = self.model.encode(texts, convert_to_numpy = True).astype(np.float32)
        return emb

    def embed_query(self, query):
        emb = self.model.encode([query], convert_to_numpy = True).astype(np.float32)
        return emb

#---------------------------------------------------------------------------------#
class BaseEmbeddingModel:

    def sentencetransformer(self):
        pass
    def BGE(self):
        pass
    def E5(self):
        pass
    def openaiembed(self):
        pass

In [ ]:
import faiss
import numpy as np

class VectorStore:

    def __init__(
        self,
        index_type="flat",
        n_list=100,
        nprobe=10,
        hnsw_m=16,
        hnsw_efconstruct=40,
        hnsw_efsearch=16
    ):
        self.index_type = index_type
        self.n_list = n_list
        self.nprobe = nprobe
        self.hnsw_m = hnsw_m
        self.hnsw_efconstruct = hnsw_efconstruct
        self.hnsw_efsearch = hnsw_efsearch
        self.index = None

    def add(self, embeddings):
        d = embeddings.shape[1]

        if self.index is None:
            if self.index_type == "flat":
                self.index = faiss.IndexFlatL2(d)

            elif self.index_type == "hnsw":
                self.index = faiss.IndexHNSWFlat(d, self.hnsw_m)
                self.index.hnsw.efConstruction = self.hnsw_efconstruct
                self.index.hnsw.efSearch = self.hnsw_efsearch

            elif self.index_type == "ivf":
                n_list = min(self.n_list, max(1, int(np.sqrt(len(embeddings)))))
                quantizer = faiss.IndexFlatL2(d)
                self.index = faiss.IndexIVFFlat(quantizer, d, n_list)
                self.index.train(embeddings)
                self.index.nprobe = min(self.nprobe, n_list)

            else:
                raise ValueError(f"Unknown index type: {self.index_type}")

        return self.index.add(embeddings)

    def search(self, query_embeddings, k=5):
        return self.index.search(query_embeddings, k)

    @property
    def ntotal(self):
        return self.index.ntotal

In [ ]:
#retriever class
class Retriever:

    def __init__(self, embedder, vector_store, child_to_parent, parents):
        self.embedder = embedder
        self.vector_store = vector_store
        self.child_to_parent = child_to_parent
        self.parents = parents
#parent child retrieval strategy

    def retrieve(self, query, k = 10):
        q_emb = self.embedder.embed_query(query)
        D, I = self.vector_store.search(q_emb, k)

        result = []
        parent = []
        parent_dist = {}
        for idx , dist in zip(I[0], D[0]):
            parent_id = self.child_to_parent[idx]

            if parent_id in parent_dist:
                parent_dist[parent_id] = min(parent_dist[parent_id],dist)
            else :
                parent_dist[parent_id] = dist

        parent_dist = sorted(parent_dist.items(), key = lambda x: x[1])
        for parent_id, dist in parent_dist:
            result.append(
                {
                    'parent_id': parent_id,
                    'text':self.parents[parent_id].page_content,
                    'retrieval_score':dist,
                    'metadata':{'source' :self.parents[parent_id].metadata['source'],
                                'page_num':self.parents[parent_id].metadata['page_label']
                    }
                }
            )

        return result

In [ ]:
#reranker class
class Reranker:

    def __init__(self, model = None):
        self.model = model

    def rerank(self, query, docs, top_k = 5):
        if self.model is None:
            return docs[:top_k] if top_k else docs

        else:
            pairs = [
                (query, doc['text']) for doc in docs
            ]
            scores = self.model.predict(pairs)
            for score, doc in zip(scores, docs):
                doc['rerank_score'] = score

            docs = sorted(docs, key = lambda x: x['rerank_score'], reverse= True)
            return docs[:top_k]

In [ ]:
#contextbuilder
class ContextBuilder:

    def build(self, docs):
        context_parts = []
        for i, doc in enumerate(docs, start = 1):
            context_parts.append(f"Source{i}:from {doc['metadata']['source']} \npage = {doc['metadata']['page_num']}\n{doc['text']}")
        context = f"\n{"-"*50}\n".join(context_parts)
        return context

In [ ]:
import os
import pickle
import faiss
# from rag.loader import DocumentLoader
# from rag.chunker import Chunker
# from rag.embedder import Embedder
# from rag.vectorstore import VectorStore
# from rag.retriever import Retriever


class IngestionPipeline:

    def __init__(
        self,
        loader_model,
        embedder_model,
        child_chunk_size=400,
        child_chunk_overlap=100,
        parent_chunk_size=1000,
        parent_chunk_overlap=200,
        index_type="flat",
        n_list=100,
        nprobe=10,
        hnsw_m=16,
        hnsw_efconstruct=40,
        hnsw_efsearch=16
    ):
        self.loader = DocumentLoader(loader_model)
        self.chunker = Chunker(
            child_chunk_size=child_chunk_size,
            child_chunk_overlap=child_chunk_overlap,
            parent_chunk_size=parent_chunk_size,
            parent_chunk_overlap=parent_chunk_overlap
        )
        self.embedder = Embedder(embedder_model)
        self.vector_store = VectorStore(
            index_type=index_type,
            n_list=n_list,
            nprobe=nprobe,
            hnsw_m=hnsw_m,
            hnsw_efconstruct=hnsw_efconstruct,
            hnsw_efsearch=hnsw_efsearch
        )

    def ingest(self, paths):
        all_docs = []
        for path in paths:
            all_docs.extend(self.loader.load(path))

        documents = all_docs
        parents, children, child_to_parent = self.chunker.split(documents)

        texts = [doc.page_content for doc in children.values()]
        embeddings = self.embedder.embed_docs(texts)
        self.vector_store.add(embeddings)

        retriever = Retriever(
            self.embedder,
            self.vector_store,
            child_to_parent,
            parents
        )

        return {
            "retriever": retriever,
            "vector_store": self.vector_store,
            "parent_chunks": parents,
            "child_to_parent": child_to_parent
        }


    def save(self, artifacts, save_dir):
        os.makedirs(save_dir, exist_ok=True)

        faiss.write_index(
            artifacts["vector_store"].index,
            os.path.join(save_dir, "index.faiss")
        )

        with open(os.path.join(save_dir, "parents.pkl"),"wb") as f:
            pickle.dump(artifacts["parent_chunks"], f)

        with open(os.path.join(save_dir, "child_to_parent.pkl"),"wb") as f:
            pickle.dump(artifacts["child_to_parent"], f)

        print(f"Artifacts saved to {save_dir}")

    def load(self, save_dir):
        index = faiss.read_index(os.path.join(save_dir, "index.faiss"))

        with open(os.path.join(save_dir, "parents.pkl"),"rb") as f:
            parents = pickle.load(f)

        with open(os.path.join(save_dir, "child_to_parent.pkl"),"rb") as f:
            child_to_parent = pickle.load(f)

        vector_store = VectorStore(index_type=self.vector_store.index_type)
        vector_store.index = index

        retriever = Retriever(
            self.embedder,
            vector_store,
            child_to_parent,
            parents
        )

        return {
            "retriever": retriever,
            "vector_store": vector_store,
            "parent_chunks": parents,
            "child_to_parent": child_to_parent
        }

In [ ]:
#generator class
class Generate:
    def __init__(self, llm):
        self.llm = llm

    def generate(self, query, context):
        prompt = f"""
                context :  {context}


                query :{query}?
                    """

        messages = f"""
        <|im_start|>system
        You are Dolphin, a helpful Q&A assistant, \nreply from given context,\nif answer is not in context reply with 'I don't Know'\n<|im_end|>
        <|im_start|>user
        {prompt}<|im_end|>
        <|im_start|>assistant
        """

        answer = self.llm(messages, max_new_tokens= 300)[0]["generated_text"]
        answer = answer[len(messages) : ].strip()
        return answer

In [ ]:
#ragpipeline class
class RAGPipeline:

    def __init__(
        self,
        retriever,
        reranker,
        context_builder,
        generator
    ):
        self.retriever = retriever
        self.reranker = reranker
        self.context_builder = context_builder
        self.generator = generator

    def run(self, query, retrieve_k=10, rerank_k=3):

        docs = self.retriever.retrieve(
            query=query,
            k=retrieve_k
        )
        docs = self.reranker.rerank(
            query=query,
            docs=docs,
            top_k=rerank_k
        )
        context = self.context_builder.build(
            docs
        )
        answer = self.generator.generate(
            query=query,
            context=context
        )

        return {
            "query": query,
            "documents": docs,
            "context": context,
            "answer": answer
        }

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from sentence_transformers import SentenceTransformer

emb_model = SentenceTransformer("microsoft/harrier-oss-v1-270m", model_kwargs={"dtype": "auto"})

ranker_model = "BAAI/bge-reranker-base"

from transformers import pipeline
llm = pipeline(
    "text-generation",
    model="dphn/Dolphin3.0-Qwen2.5-0.5B"
)

/tmp/ipykernel_2413/3483559541.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/7.61k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/7.36k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/759 [00:00<?, ?B/s]

In [ ]:
import requests
pdf1 = requests.get(url = "https://arxiv.org/pdf/1706.03762").content
with open("attention.pdf", 'wb') as f:
    f.write(pdf1)
pdf2 = requests.get(url = "https://arxiv.org/pdf/1810.04805").content
with open("bert.pdf", 'wb') as f:
    f.write(pdf2)

ingestion = IngestionPipeline(
    loader_model=PyPDFLoader,
    embedder_model=emb_model,
    index_type="hnsw"
)

artifacts = ingestion.ingest(
    ["/content/attention.pdf", "/content/bert.pdf"]
)

retriever = artifacts["retriever"]
vector_store = artifacts["vector_store"]

In [ ]:
rag = RAGPipeline(retriever = retriever,
                  reranker = Reranker(),
                  context_builder = ContextBuilder(),
                  generator = Generate(llm))

In [ ]:
rag.run("what is self-attention?")['documents'][0]['metadata']

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'source': '/content/attention.pdf', 'page_num': '2'}

In [ ]:
ingestion.save(
    artifacts,
    "attention_store"
)

Artifacts saved to attention_store


In [ ]:
class RetrievalEvaluator:

    def __init__(self, retriever):
        self.retriever = retriever

    def hit_rate(self, dataset, k=5):

        hits = 0

        for sample in dataset:

            query = sample["question"]
            expected_page = str(sample["expected_page"]) # Convert to string for comparison

            docs = self.retriever.retrieve(
                query,
                k=k
            )

            pages = [
                doc["metadata"]["page_num"]
                for doc in docs
            ]

            if expected_page in pages:
                hits += 1

        return hits / len(dataset)

    def recall_at_k(self, dataset, k=5):

        hits = 0

        total = len(dataset)

        for sample in dataset:

            query = sample["question"]
            expected_page = str(sample["expected_page"]) # Convert to string for comparison

            docs = self.retriever.retrieve(
                query,
                k=k
            )

            pages = [
                doc["metadata"]["page_num"]
                for doc in docs
            ]

            if expected_page in pages:
                hits += 1

        return hits / total

    def evaluate(self, dataset, k=5):

        return {
            "Hit@K": self.hit_rate(
                dataset,
                k
            ),
            "Recall@K": self.recall_at_k(
                dataset,
                k
            )
        }

In [ ]:
eval_set = [
    {
        "question": "What is bert stands for?",
        "expected_page": 1
    },
    {
        "question": "what is self attention",
        "expected_page" : 1
    }
]

In [ ]:
evaluator = RetrievalEvaluator(
    retriever
)

results = evaluator.evaluate(
    eval_set,
    k=5
)

print(results)

{'Hit@K': 0.5, 'Recall@K': 0.5}
